# Lesson 5: Image Moments

Once we can isolate a blob (Lesson 4), moments let us summarize its shape with a handful of numbers: its area, centroid, orientation, and even a description that stays the same under translation, scale, and rotation. This is a classic, lightweight alternative to learned features for simple shape matching.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## What is a moment?

For a binary image, the raw moment $M_{ij}$ is defined as

$$M_{ij} = \sum_{x,y} x^i y^j \, I(x, y)$$

where $I(x,y)$ is 1 on the shape and 0 elsewhere. A few special cases are already familiar quantities:

- $M_{00}$ = area (pixel count)
- $\bar{x} = M_{10}/M_{00}$, $\bar{y} = M_{01}/M_{00}$ = centroid

`cv2.moments` computes all of these (plus *central* moments $\mu_{ij}$, which are translation-invariant, and Hu moments, which are translation-, scale-, and rotation-invariant) in one call.

## An elongated, rotated blob

We draw a rotated ellipse so its orientation is easy to eyeball and check against what the moments compute.

In [ ]:
binary = np.zeros((200, 200), dtype=np.uint8)
center = (100, 100)
axes_len = (70, 25)
angle_deg = 30
cv2.ellipse(binary, center, axes_len, angle_deg, 0, 360, 255, -1)

plt.imshow(binary, cmap='gray')
plt.title(f'Ellipse drawn at {angle_deg} degrees')
plt.axis('off')
plt.show()

## Area and centroid from moments

In [ ]:
m = cv2.moments(binary, binaryImage=True)

area = m['m00']
cx = m['m10'] / m['m00']
cy = m['m01'] / m['m00']

print(f'area (m00)   = {area:.0f} pixels')
print(f'centroid     = ({cx:.1f}, {cy:.1f})')

plt.imshow(binary, cmap='gray')
plt.scatter(cx, cy, c='red', marker='x', s=80)
plt.title('Centroid from moments')
plt.axis('off')
plt.show()

## Orientation from central moments

The central moments $\mu_{20}$, $\mu_{02}$, $\mu_{11}$ describe the spread of the shape around its centroid — essentially its covariance matrix. The angle of the major axis (the direction of greatest spread) is

$$\theta = \frac{1}{2}\,\mathrm{atan2}\!\left(2\mu_{11},\; \mu_{20} - \mu_{02}\right)$$

In [ ]:
theta = 0.5 * np.arctan2(2 * m['mu11'], m['mu20'] - m['mu02'])
theta_deg = np.degrees(theta)

print(f'orientation from moments = {theta_deg:.1f} degrees')
print(f'angle used to draw the ellipse = {angle_deg} degrees')

# The eigenvalues of the (normalized) covariance matrix of central moments also
# give the semi-axis lengths of the equivalent ellipse: semi_axis = 2*sqrt(eigenvalue).
# Using the true length (rather than an arbitrary one) makes the plotted line end
# exactly at the ellipse's tips instead of overshooting past the curved boundary,
# which is what previously made it look slightly misaligned near the tip.
cov = np.array([[m['mu20'], m['mu11']], [m['mu11'], m['mu02']]]) / m['m00']
eigvals, _ = np.linalg.eigh(cov)
semi_major = 2 * np.sqrt(eigvals[-1])

dx, dy = semi_major * np.cos(theta), semi_major * np.sin(theta)

plt.imshow(binary, cmap='gray')
plt.plot([cx - dx, cx + dx], [cy - dy, cy + dy], c='red', linewidth=2)
plt.scatter(cx, cy, c='red', marker='x', s=80)
plt.title('Major axis recovered from moments')
plt.axis('off')
plt.axis('equal')
plt.show()

## Hu moments: a shape descriptor invariant to pose

`cv2.HuMoments` combines the central moments into 7 values that stay (nearly) the same regardless of the shape's position, size, and rotation. This makes them useful for comparing two shapes without first aligning them.

To see this, we build the same ellipse three times: at the original pose, translated, and rotated + scaled.

In [ ]:
def make_ellipse(canvas_size, center, axes_len, angle_deg):
    img = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
    cv2.ellipse(img, center, axes_len, angle_deg, 0, 360, 255, -1)
    return img

def hu_log(img):
    m = cv2.moments(img, binaryImage=True)
    hu = cv2.HuMoments(m).flatten()
    # log-scale since raw Hu moments span many orders of magnitude
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)

original = make_ellipse(200, (100, 100), (70, 25), 30)
translated = make_ellipse(200, (60, 140), (70, 25), 30)
rotated_scaled = make_ellipse(200, (100, 100), (35, 12), 110)
different_shape = make_ellipse(200, (100, 100), (50, 45), 0)  # a much rounder ellipse

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, img, title in zip(
    axes,
    [original, translated, rotated_scaled, different_shape],
    ['Original', 'Translated', 'Rotated + scaled', 'Different shape'],
):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
hu_original = hu_log(original)

print(f'{"image":>18} {"L2 distance to original Hu moments":>36}')
for img, name in [
    (original, 'original'),
    (translated, 'translated'),
    (rotated_scaled, 'rotated+scaled'),
    (different_shape, 'different shape'),
]:
    dist = np.linalg.norm(hu_log(img) - hu_original)
    print(f'{name:>18} {dist:>36.3f}')

Translated and rotated+scaled versions land close to zero distance — the Hu moments barely changed even though the pixels look very different. The genuinely different shape (a rounder ellipse) stands out with a much larger distance. This is exactly the invariance property that makes Hu moments useful for shape matching.

### Exercise

1. Use `cv2.findContours` to get the outline of a blob from Lesson 4's binary image, then call `cv2.moments` on the *contour* instead of the full binary mask. Compare the centroid to the one computed here.
2. Draw a shape that is mirror-flipped rather than rotated. Are its Hu moments still close to the original? (Hint: think about what determinant/parity information moments do or don't capture.)